In [2]:
import os
import glob
import random
import pandas as pd
import numpy as np
import torch  
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from numba import njit, float32, int64, types
from numba.typed import Dict
from tqdm import tqdm
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# [수정 필요] 데이터셋 최상위 경로 (이 안에 DoS, Spoofing 등 폴더가 있어야 함)
BASE_PATH = "C:/Users/user/Desktop/IDS_masters/CarHacking_Dataset/9) Car-Hacking Dataset"

# [수정 필요] 저장할 파일 경로
PT_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/validation_dataset.pt"
CSV_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/validation_dataset_flattened.csv"

WINDOW_SIZE = 64
STRIDE = 32

FEATURE_NAMES = [
    "1. Global IAT", "2. ID IAT", "3. Entropy", 
    "4. Hamming", "5. DLC", "6. Delta", 
    "7. Jitter", "8. Byte Mean", "9. Byte Std"
]

In [5]:
# ==========================================
# 2. 헬퍼 함수 (ID 파싱, Payload 파싱)
# ==========================================
def parse_id(id_val):
    if isinstance(id_val, str):
        try:
            return int(id_val, 16)
        except:
            return 0
    return int(id_val)

def parse_payload_str(s, max_len=8):
    """'00 00 A1 ...' 형태의 문자열을 길이 8의 리스트로 변환"""
    parts = str(s).split()
    vals = []
    for p in parts:
        if p != "":
            try:
                vals.append(int(p, 16))
            except:
                pass
    
    if len(vals) < max_len:
        vals += [0] * (max_len - len(vals))
    return vals[:max_len]

In [6]:
# ==========================================
# 3. Numba 피처 계산 로직 (수정 없음, 그대로 사용)
# ==========================================
@njit(fastmath=True)
def calculate_features_numba(timestamps, can_ids, dlcs, payloads):
    n = len(timestamps)
    features = np.zeros((n, 9), dtype=np.float32)
    
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    
    prev_global_time = timestamps[0]

    for i in range(n):
        ts = timestamps[i]
        cid = can_ids[i]
        
        # 1. Global IAT
        if i == 0: g_iat = 0.0
        else: g_iat = ts - prev_global_time
        if g_iat > 0.1: g_iat = 0.1
        features[i, 0] = float32(g_iat / 0.1)
        prev_global_time = ts
        
        # 2. ID IAT
        if cid in last_time_map: id_iat = ts - last_time_map[cid]
        else: id_iat = 0.001
        features[i, 1] = float32(np.log1p(id_iat * 1000.0) / 7.0) 
        
        # 7. Jitter
        if cid in last_iat_map: jitter = np.abs(id_iat - last_iat_map[cid])
        else: jitter = 0.0
        if jitter > 0.05: jitter = 0.05
        features[i, 6] = float32(jitter / 0.05)
        
        last_time_map[cid] = ts
        last_iat_map[cid] = id_iat

        curr_p_val = 0
        for b in range(8): curr_p_val = (curr_p_val << 8) | payloads[i, b]
            
        # 4. Hamming & 6. Delta
        if cid in last_payload_map:
            prev_p_val = last_payload_map[cid]
            xor_val = curr_p_val ^ prev_p_val
            ham_dist = 0
            temp_xor = xor_val
            while temp_xor > 0:
                if temp_xor & 1: ham_dist += 1
                temp_xor >>= 1
            
            if curr_p_val > prev_p_val: delta = curr_p_val - prev_p_val
            else: delta = prev_p_val - curr_p_val
        else:
            ham_dist = 0
            delta = 0
            
        features[i, 3] = float32(ham_dist / 64.0)
        features[i, 5] = float32(np.log1p(float(delta)) / 45.0) 
        last_payload_map[cid] = curr_p_val

        # 3. Entropy & 8. Mean & 9. Std
        counts = np.zeros(256, dtype=np.int32)
        row = payloads[i]
        s = 0.0
        for b_idx in range(8):
            val = row[b_idx]
            counts[val] += 1
            s += val
            
        ent = 0.0
        for c in counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)
        features[i, 2] = float32(ent / 2.1)

        features[i, 4] = float32(dlcs[i] / 8.0) # 5. DLC

        mean_val = s / 8.0
        features[i, 7] = float32(mean_val / 255.0)
        
        var_s = 0.0
        for b_idx in range(8):
            diff = row[b_idx] - mean_val
            var_s += diff * diff
        std_val = np.sqrt(var_s / 8.0)
        features[i, 8] = float32(std_val / 128.0)

    return features

In [9]:
def main():
    print("=== main start ===")
    LABEL_MAP = {
        "DoS": 1,
        "Fuzzing": 2,
        "Spoofing": 4, "RPM": 4, "Gear": 4, 
        "Replay": 3
    }
    
    # 1. 파일 목록 가져오기
    csv_files = glob.glob(os.path.join(BASE_PATH, "**", "*.csv"), recursive=True)
    txt_files = glob.glob(os.path.join(BASE_PATH, "**", "*.txt"), recursive=True)
    all_files = csv_files + txt_files
    
    if not all_files:
        print(f"[ERROR] 파일을 찾을 수 없습니다: {BASE_PATH}")
        return

    print(f"[INFO] 발견된 파일: {len(all_files)}개")

    # 🔹 여기서부터는 '파일별 윈도우'를 모으는 전역 리스트
    X_list_total = []
    y_list_total = []

    # 패킷 단위 feature CSV용
    packet_features_all = []
    packet_labels_all = []

    for file in all_files:
        file_name = os.path.basename(file)
        full_path = str(file)
        ext = os.path.splitext(file)[1].lower()
        
        print(f"📂 처리 중: {file_name}")
        
        try:
            current_df = None
            
            # ---------------------------------------------------------
            # CASE A: .txt 파일
            # ---------------------------------------------------------
            if ext == '.txt':
                txt_data_list = []
                with open(file, 'r', encoding='utf-8') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 7:
                            continue
                        try:
                            # 0:Timestamp:, 1:시간, 2:ID:, 3:아이디, 4:000, 5:DLC:, 6:길이, 7~:데이터
                            ts = parts[1]
                            cid = parts[3]
                            dlc = parts[6]
                            payload = " ".join(parts[7:])
                            
                            txt_data_list.append([ts, cid, dlc, payload, 0, "Normal"])
                        except:
                            continue
                
                if not txt_data_list:
                    print("   ⚠️ [SKIP] 빈 텍스트 파일이거나 형식이 맞지 않습니다.")
                    continue
                    
                current_df = pd.DataFrame(
                    txt_data_list, 
                    columns=["timestamp", "can_id", "dlc", "payload", "Label_Int", "Label_Str"]
                )

            # ---------------------------------------------------------
            # CASE B: .csv 파일
            # ---------------------------------------------------------
            else:
                temp_df = pd.read_csv(
                    file, 
                    header=None, 
                    names=[f"col_{i}" for i in range(20)], 
                    engine='python'
                )
                
                def get_last_valid_label(row):
                    vals = row.dropna()
                    if len(vals) == 0: 
                        return "0"
                    return str(vals.iloc[-1]).strip()
                
                raw_labels = temp_df.apply(get_last_valid_label, axis=1)
                
                def join_payload(row):
                    valid_vals = row.dropna()
                    if len(valid_vals) > 4:
                        payload_parts = valid_vals.iloc[3:-1]
                        return " ".join(payload_parts.astype(str))
                    return ""
                
                payload_str = temp_df.apply(join_payload, axis=1)
                
                current_df = pd.DataFrame({
                    "timestamp": temp_df.iloc[:, 0],
                    "can_id": temp_df.iloc[:, 1],
                    "dlc": temp_df.iloc[:, 2],
                    "payload": payload_str,
                    "Label_Int": 0,
                    "Label_Str": raw_labels
                })
                current_df.dropna(subset=["timestamp", "can_id"], inplace=True)

            # ---------------------------------------------------------
            # 공통: 공격 타입 라벨링 (폴더/파일명 기반)
            # ---------------------------------------------------------
            current_attack_id = 0
            for key, val in LABEL_MAP.items():
                if key.lower() in full_path.lower():
                    current_attack_id = val
                    break
            
            print(f"   -> 감지된 공격 타입: {current_attack_id}")
            
            if ext == '.txt':
                if current_attack_id != 0:
                    current_df['Label_Int'] = current_attack_id
                else:
                    current_df['Label_Int'] = 0
            else:
                new_labels = np.zeros(len(current_df), dtype=int)
                if current_attack_id != 0:
                    attack_indices = (current_df['Label_Str'] == 'T')
                    new_labels[attack_indices] = current_attack_id
                    print(f"   -> 공격 패킷('T') {np.sum(attack_indices)}개 변환 완료.")
                current_df['Label_Int'] = new_labels

            # ---------------------------------------------------------
            # 🔥 여기서부터가 핵심: "파일별"로 feature + 윈도우 생성
            # ---------------------------------------------------------
            # numpy 변환
            try:
                timestamps = current_df["timestamp"].astype(np.float64).to_numpy()
            except Exception as e:
                print(f"   ❌ timestamp 변환 실패, 스킵: {e}")
                continue

            can_ids = current_df["can_id"].apply(parse_id).astype(np.int64).to_numpy()
            dlcs = current_df["dlc"].astype(np.int64).to_numpy()
            payload_array = np.vstack(
                current_df["payload"].apply(parse_payload_str).values
            ).astype(np.uint8)
            labels_int = current_df["Label_Int"].values.astype(np.int64)

            # 패킷 수가 WINDOW_SIZE보다 작으면 윈도우는 못 만들지만, 패킷 feature는 만들 수 있음
            features_f = calculate_features_numba(timestamps, can_ids, dlcs, payload_array)

            # 패킷 단위 feature 저장 (나중에 CSV용)
            packet_features_all.append(features_f)
            packet_labels_all.append(labels_int)

            if len(features_f) < WINDOW_SIZE:
                print(f"   ⚠️ [SKIP] 윈도우 생성 불가 (패킷 {len(features_f)} < WINDOW_SIZE {WINDOW_SIZE})")
                continue

            # ✔ 이 파일 내부에서만 윈도우 슬라이딩
            for start in range(0, len(features_f) - WINDOW_SIZE + 1, STRIDE):
                end = start + WINDOW_SIZE
                X_list_total.append(features_f[start:end].T)  # (64, 9) 같은 형태
                y_list_total.append(labels_int[start:end])

            print(f"   -> 생성된 윈도우 수: {(len(features_f) - WINDOW_SIZE) // STRIDE + 1}")

        except Exception as e:
            print(f"   ❌ 파일 처리 실패: {e}")
            continue

    # ---------------------------------------------------------
    # 🔚 루프 종료 후: 전체 윈도우 / 패킷 feature 통합 및 저장
    # ---------------------------------------------------------
    if not X_list_total:
        print("데이터 없음 (윈도우가 하나도 생성되지 않았습니다).")
        return

    X = np.stack(X_list_total, axis=0)   # (num_windows, T, F) 또는 (num_windows, F, T)
    y = np.stack(y_list_total, axis=0)   # (num_windows, T)

    print(f"[RESULT] 생성된 윈도우: {X.shape}, 라벨: {y.shape}")

    NPY_SAVE_PATH = PT_SAVE_PATH.replace(".pt", ".npz")
    np.savez(NPY_SAVE_PATH, X=X.astype(np.float32), y=y.astype(np.int64))
    print(f"[DONE] .npz(numpy) 저장 완료: {NPY_SAVE_PATH}")


if __name__ == "__main__":
    main()


=== main start ===
[INFO] 발견된 파일: 5개
📂 처리 중: DoS_dataset.csv
   -> 감지된 공격 타입: 1
   -> 공격 패킷('T') 587521개 변환 완료.
   -> 생성된 윈도우 수: 114554
📂 처리 중: Fuzzy_dataset.csv
   -> 감지된 공격 타입: 2
   -> 공격 패킷('T') 491847개 변환 완료.
   -> 생성된 윈도우 수: 119963
📂 처리 중: gear_dataset.csv
   -> 감지된 공격 타입: 4
   -> 공격 패킷('T') 597252개 변환 완료.
   -> 생성된 윈도우 수: 138847
📂 처리 중: RPM_dataset.csv
   -> 감지된 공격 타입: 4
   -> 공격 패킷('T') 654897개 변환 완료.
   -> 생성된 윈도우 수: 144427
📂 처리 중: normal_run_data.txt
   -> 감지된 공격 타입: 0
   -> 생성된 윈도우 수: 30901
[RESULT] 생성된 윈도우: (548692, 9, 64), 라벨: (548692, 64)
[DONE] .npz(numpy) 저장 완료: C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/validation_dataset.npz
